# 230 · e-f Rabi on a shared drive port — solution

**Level:** medium · **Topics:** Gen2, SHFQC, dsl, sections, timing, calibration, sweeps · **Type:** write the code

Solution of [exercise.ipynb](exercise.ipynb).

In [ ]:
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation
from laboneq.simple import *

## Device setup

Two logical signals, `drive` and `drive_ef`, are connected to the same port.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/drive_ef", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Step 1: calibration

The g-e transition is at 4.10 GHz. The e-f transition is at **3.93 GHz**.

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html) ·
[Calibration properties](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/02_calibration_properties.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.0e9),
    range=10,
)
calibration["q0/drive_ef"] = SignalCalibration(
    oscillator=Oscillator(frequency=-70e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.0e9),
    range=10,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=80e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=-25,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=80e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=10,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Step 2: the sequence

For every amplitude: g-e pulse on `drive`, then e-f pulse on `drive_ef` with the swept amplitude, then the readout.

📖 [Section tutorial](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/00_section_tutorial.html) ·
[Averaging and sweeping](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/04_averaging_sweeping.html)

In [ ]:
ge_pulse = pulse_library.gaussian(length=23e-9, amplitude=0.7)
ef_pulse = pulse_library.gaussian(length=16e-9, amplitude=1.0)
readout_pulse = pulse_library.const(length=2e-6, amplitude=0.5)

amplitudes = LinearSweepParameter(start=0.1, stop=1.0, count=10)

In [ ]:
@dsl.experiment(signals=["drive", "drive_ef", "measure", "acquire"])
def ef_rabi():
    with dsl.acquire_loop_rt(count=10):
        with dsl.sweep(parameter=amplitudes) as amplitude:
            with dsl.section(
                name="ge_pulse",
                alignment=SectionAlignment.RIGHT,
                on_system_grid=True,
            ) as ge_section:
                dsl.play("drive", ge_pulse)
            with dsl.section(
                name="ef_pulse",
                play_after=ge_section,
                on_system_grid=True,
            ) as ef_section:
                dsl.play("drive_ef", ef_pulse, amplitude=amplitude)
            with dsl.section(name="readout", play_after=ef_section):
                dsl.measure(
                    measure_signal="measure",
                    acquire_signal="acquire",
                    measure_pulse=readout_pulse,
                    integration_kernel=readout_pulse,
                    handle="result",
                    reset_delay=1e-6,
                )


experiment = ef_rabi()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "drive_ef": "q0/drive_ef",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

## Step 3: compile and run

One output has one hardware oscillator active at a time. Between the two pulses the instrument must switch
from the g-e to the e-f oscillator, and it can only do that on the **system grid** (8 ns on the SHFQC+).
The g-e pulse is 23 ns long, so the e-f pulse would start off the grid.

- `on_system_grid=True` on the e-f section moves its start to the grid: 24 ns.
- `alignment=SectionAlignment.RIGHT` on the g-e section pushes the g-e pulse against the e-f pulse, so there is no 1 ns gap between them.

📖 [Timing rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/01_timing_rules.html)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

In [ ]:
# check
print(results["result"].data.shape)

**Expected result:** `(10,)`

Look at the first shot. Both pulses are on the same output: the g-e pulse ends at 24 ns, the e-f pulse runs from 24 ns to 40 ns, and the readout starts at 40 ns.

In [ ]:
plot_simulation(compiled, start_time=0, length=150e-9, signals=["drive", "measure"])

## Answers

1. The output plays one hardware oscillator at a time. Switching from the g-e to the e-f oscillator is only possible on the system grid, every 8 ns. 23 ns is not on the grid.
2. 24 ns is a multiple of 8 ns, so the e-f pulse starts on the grid by itself. This is why such a sequence can work for months and fail after a small change of a pulse length.
3. No. A port has one local oscillator, so both signals must use the same value and differ only in the IF. LabOne Q does not check this here: with two different values the experiment still compiles, and one of the two transitions is driven at the wrong frequency.
4. Yes, it compiles without the system grid: the modulation is then computed into the waveforms and no oscillator switch is needed. The price is one waveform per frequency, and no real-time frequency sweeps.

**Takeaway:** two signals on one port share the local oscillator, and a change of hardware oscillator needs the system grid.